# Matryoshka vs. vanilla SAEs on a known hierarchy

A sparse autoencoder trained on a flat dictionary has no reason to keep a general
concept and its special cases apart. When a parent feature fires on every token its
children fire on, one broad latent covering the whole family reconstructs the input
about as well as four precise ones — and pays less sparsity penalty for it. The
hierarchy collapses.

A **Matryoshka** SAE trains nested prefixes of the dictionary at the same time. Each
step samples several prefix lengths and scores the reconstruction of *each* prefix, so
the first few latents must stand on their own without the later ones available to help
(`sample_prefixes` in `sae.py`, biased toward short prefixes; `permute_latents` keeps
the highest-contribution latents drifting toward the front). Broad structure is forced
early, detail is pushed outward. **Vanilla is the same model with `n_prefixes=1`** —
one prefix spanning the whole dictionary — which is why both are trained here from the
same class and differ in exactly two config entries.

This notebook builds a world where the hierarchy is known exactly, trains both SAEs on
it, and then asks two different questions:

| | question | what it scores |
| --- | --- | --- |
| §7 | **feature recovery** | did the SAE find the ground-truth directions at all? |
| §8 | **edge recovery** | do the hierarchy metrics recover the known parent→child tree? |

The second is the Exp-0 question. The toy is where those metrics get calibrated,
because it is the only setting where the right answer is known in advance — on
`gemma-2-2b` there is nothing to check an answer against.


## Setup

Runs against the upstream [matryoshka-saes](https://github.com/noanabeshima/matryoshka-saes)
repo, which the next cell clones into the `sae-training` checkout beside this one
(that is also where the checkpoints land, so a re-run finds the previous one).

This notebook used to live in `sae-training/scripts/`. It now sits in the metrics
repo, beside the Tier-1 notebook, so the next cell anchors on **`metrics/`** — the
directory holding `config.py` — and finds `sae-training/` from there. Set
`EXP0_SAE_TRAINING` if your clone is somewhere else; that is the same environment
variable `validation/calibrate_on_trained_toy.py` reads.

Any kernel with the training dependencies will do:

```
uv venv --python 3.12 .venv
VIRTUAL_ENV=.venv uv pip install torch numpy tqdm jsonschema nbformat plotly ipython scipy kaleido ipykernel
.venv/bin/python -m ipykernel install --user --name sae-training --display-name "Python (sae-training)"
```

In [1]:
import os
import subprocess
import sys
from pathlib import Path

# Anchor on the metrics repo root -- the directory holding config.py -- rather than
# counting `.parent` hops, so moving this notebook does not break anything below.
# It used to anchor on sae-training/pyproject.toml, which stopped resolving the
# moment the notebook moved out of that repo. Idempotent: safe to re-run after the
# os.chdir at the end.
_start = Path.cwd().resolve()
METRICS_ROOT = next(p for p in [_start, *_start.parents] if (p / "config.py").is_file())
WORKSPACE = METRICS_ROOT.parent          # holds sae-training/, metrics/, PCFG/

# sae-training is a separate repo, so where it sits is the user's choice. Same
# search order (and same env var) as validation/calibrate_on_trained_toy.py.
_env = os.environ.get("EXP0_SAE_TRAINING")
_candidates = [Path(_env)] if _env else [WORKSPACE / "sae-training",
                                         METRICS_ROOT / "sae-training"]
SAE_TRAINING = next((c for c in _candidates if (c / "configs" / "tree.json").is_file()),
                    _candidates[0])
if not SAE_TRAINING.is_dir():
    raise SystemExit(
        "cannot find the sae-training checkout (looked in: "
        + ", ".join(str(c) for c in _candidates)
        + ").\nClone https://github.com/soar-eleuther-i6-hierarchy/sae-training "
        "beside metrics/, or set EXP0_SAE_TRAINING to its path.")

# The upstream clone has lived in both places; take whichever exists.
repo = next((c for c in (SAE_TRAINING / "matryoshka-saes",
                         SAE_TRAINING / "configs" / "matryoshka-saes") if c.exists()),
            SAE_TRAINING / "matryoshka-saes")

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/noanabeshima/matryoshka-saes.git", str(repo)],
        check=True,
    )

# toy_model.py reads ./tree.json and ./tree.schema.json at import time, so the repo
# has to be both importable and the working directory.
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
os.chdir(repo)

# Figures land here, NOT in the upstream clone. `os.chdir(repo)` below makes the
# clone the working directory, so the relative "figures/..." paths this notebook
# used to export to resolved inside a third-party checkout -- overwriting files
# that repo tracks, and putting our results somewhere a re-clone would erase.
FIG_DIR = METRICS_ROOT / "validation" / "figures" / "tier2"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("metrics:", METRICS_ROOT)
print("sae-training:", SAE_TRAINING)
print("upstream repo:", repo)
print("cwd:", Path.cwd())

metrics: /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/metrics
sae-training: /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/sae-training
upstream repo: /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/sae-training/matryoshka-saes
cwd: /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/sae-training/matryoshka-saes


## 1. The toy world

`tree.json` describes a generative process, not a neural network. Each node is a
**feature** with a firing probability, and the tree imposes two rules the sampler
enforces:

1. **containment** — a child can only fire on steps where its parent fires;
2. **mutual exclusion** — where a node is marked `mutually_exclusive_children`, at most
   one of its children fires at a time.

`active_prob` (shown as `p` below) is **conditional on the parent being active**, not an
absolute firing rate — the schema defines it as *"probability this feature is active
conditional on its parent being active"*. So feature 12 with `p=0.05` hanging off a root
with `p=1.0` fires on 5% of samples, while feature 1 with `p=0.2` under a parent with
`p=0.15` fires on only 3%. The root's `p=1.0` simply means it is always active: it has no
parent to be conditioned on, and being `is_read_out: false` it is not a feature at all —
just the point the rest of the tree hangs from.

One subtlety inside an exclusive group: there `p` acts as a **pick weight**, not a coin
flip. The sampler chooses one sibling with `multinomial` weighted by `p` and that sibling
then fires with certainty. It still comes out equal to the conditional probability the
schema promises, but only because each group's weights sum to exactly 1.0
(`0.2 + 0.2 + 0.2 + 0.4`) — `multinomial` normalises, so weights summing to anything else
would break that correspondence.

A data point is drawn by walking the tree, then the binary firing vector is multiplied
by a feature-direction matrix to produce the activation vector the SAE sees. That is
the whole data-generating process — `TreeDataset` in `toy_model.py`.

Two details matter for everything downstream. A node with `is_read_out: false` shapes
the sampling but has **no index**, so it never appears in the data and can never be an
endpoint of a scoreable edge. And the config below sets `n_latents = tree.n_features`,
i.e. the dictionary is exactly as large as the truth — no room to hide, which is what
makes a missed feature meaningful.


In [2]:
import torch
import numpy as np
from toy_model import Tree, TreeDataset
import json


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tree_dict = json.load(open('./tree.json', 'r'))
tree = Tree(tree_dict=tree_dict)

D_MODEL = tree.n_features


def true_edges(node, edges=None):
    """Parent -> child pairs over READ-OUT features only.

    Nodes with no index carry structure but cannot be an endpoint, which is why the
    links drawn below do not all become scoreable edges.
    """
    edges = set() if edges is None else edges
    if node.index is not False:
        for child in node.children:
            if child.index is not False:
                edges.add((node.index, child.index))
    for child in node.children:
        true_edges(child, edges)
    return edges


TRUE_EDGES = sorted(true_edges(tree))
print(f"{tree.n_features} read-out features, {len(TRUE_EDGES)} scoreable edges")


20 read-out features, 9 scoreable edges


In [3]:
import plotly.graph_objects as go

BLUE, GREY, ORANGE = "#3b6ea5", "#9aa0a6", "#e8833a"
GREEN, RED, PURPLE = "#2e8b57", "#c0392b", "#7d3c98"


def tree_layout(root):
    """Left-to-right positions for every node, plus the parent->child links.

    Depth runs along x and siblings stack down y, rather than the other way round:
    20 leaves laid out horizontally need ~1200px, which a notebook output pane crops
    instead of scrolling. Stacked vertically the figure stays inside any pane width
    and grows downward, which notebooks do scroll.

    Shared by the ground-truth drawing and the recovery drawing below, so the two
    figures are directly comparable node for node.
    """
    nodes, links, cursor = [], [], [0]

    def place(node, depth):
        """Assign this node a slot and return it. Slots are handed out to leaves in
        order; a parent takes the midpoint of the span its children occupy (midpoint,
        not mean, so a node with many children on one side is not dragged off-centre).
        Only at the end is (depth, slot) turned into a coordinate."""
        if node.children:
            kid_slots = [place(c, depth + 1) for c in node.children]
            slot = (min(kid_slots) + max(kid_slots)) / 2
        else:
            kid_slots, slot = [], cursor[0]
            cursor[0] += 1

        here = (depth, -slot)
        nodes.append((here, node))
        links.extend((here, (depth + 1, -s), node, c)
                     for s, c in zip(kid_slots, node.children))
        return slot

    place(root, 0)
    return nodes, links, cursor[0]          # cursor[0] = number of leaf rows


def _arc(p0, p1, bow=0.22, n_points=32):
    """Quadratic Bezier between two nodes, bowed perpendicular to the straight line."""
    (x0, y0), (x1, y1) = p0, p1
    dx, dy = x1 - x0, y1 - y0
    length = (dx * dx + dy * dy) ** 0.5 or 1.0
    cx = (x0 + x1) / 2 - dy / length * bow * length
    cy = (y0 + y1) / 2 + dx / length * bow * length
    ts = [i / (n_points - 1) for i in range(n_points)]
    return ([(1 - t) ** 2 * x0 + 2 * (1 - t) * t * cx + t * t * x1 for t in ts],
            [(1 - t) ** 2 * y0 + 2 * (1 - t) * t * cy + t * t * y1 for t in ts])


def _legend(fig, lines=(), markers=()):
    for name, style in lines:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=name, line=style))
    for name, marker in markers:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name, marker=marker))


def _finish(fig, title, nodes, n_cols):
    """Size the canvas from the tree itself.

    Width is fixed narrow enough for any notebook pane, and height grows with the
    number of leaves, so the figure is never cropped sideways. The explicit axis
    ranges keep a marker at the edge from being clipped.
    """
    depth = max(xy[0] for xy, _ in nodes)
    fig.update_layout(
        title=title,
        width=760,                                  # fits a notebook pane; never cropped
        height=int(185 + 34 * n_cols),              # grows downward instead
        xaxis=dict(visible=False, range=[-0.35, depth + 0.35]),
        yaxis=dict(visible=False, range=[-n_cols + 0.4, 0.6]),
        # b leaves room for the legend, which wraps to two rows on the wider key
        plot_bgcolor="white", margin=dict(l=20, r=20, t=55, b=105),
        legend=dict(orientation="h", yanchor="top", y=-0.02, x=0))
    return fig


def draw_tree(root, edge_set):
    """The ground truth. Solid orange links are the scoreable edges in `edge_set`;
    dashed grey links touch a node that is not read out and so cannot be scored."""
    nodes, links, n_cols = tree_layout(root)
    fig = go.Figure()

    for (x0, y0), (x1, y1), parent, child in links:
        scoreable = (parent.index, child.index) in edge_set
        fig.add_trace(go.Scatter(
            x=[x0, x1], y=[y0, y1], mode="lines", hoverinfo="skip", showlegend=False,
            line=dict(width=2.5 if scoreable else 1,
                      color=ORANGE if scoreable else GREY,
                      dash="solid" if scoreable else "dot")))

    for readout in (True, False):
        pts = [(xy, n) for xy, n in nodes if (n.index is not False) == readout]
        if not pts:
            continue
        fig.add_trace(go.Scatter(
            x=[p[0][0] for p in pts], y=[p[0][1] for p in pts],
            mode="markers+text", showlegend=False,
            text=[str(n.index) if readout else "" for _, n in pts],
            textposition="middle center",
            textfont=dict(color="white" if readout else GREY, size=11),
            marker=dict(size=26, color=BLUE if readout else "white",
                        line=dict(width=1.5, color=BLUE if readout else GREY)),
            hovertext=[f"P(active | parent active) = {n.active_prob}"
                       + ("  (exclusive children)" if n.mutually_exclusive_children else "")
                       for _, n in pts],
            hoverinfo="text"))

    _legend(fig,
            lines=[("scoreable edge", dict(color=ORANGE, width=2.5)),
                   ("link to a node that is not read out", dict(color=GREY, width=1, dash="dot"))],
            markers=[("read-out feature", dict(size=13, color=BLUE, line=dict(width=1.5, color=BLUE))),
                     ("not read out", dict(size=13, color="white", line=dict(width=1.5, color=GREY)))])

    return _finish(fig, f"Toy hierarchy — {root.n_features} features, "
                        f"{len(edge_set)} scoreable edges", nodes, n_cols)


draw_tree(tree, set(TRUE_EDGES)).show()


In [4]:
def tree_text(node, indent=0):
    """The tree as exact text. `[--]` marks a node that shapes sampling but is not
    read out, so it holds no feature index and can end no edge.

    Returns rather than prints, so the summary figure below can place the very same
    listing beside the drawings instead of keeping a second copy of the formatting.
    """
    tag = f"[{node.index:>2}]" if node.index is not False else "[--]"
    excl = "   (children mutually exclusive)" if node.mutually_exclusive_children else ""
    lines = ["  " * indent + f"{tag}  p={node.active_prob}{excl}"]
    for child in node.children:
        lines.append(tree_text(child, indent + 1))
    return "\n".join(lines)


print(tree_text(tree))
print(f"\nscoreable edges ({len(TRUE_EDGES)}):")
print(TRUE_EDGES)


[--]  p=1.0
  [ 0]  p=0.15   (children mutually exclusive)
    [ 1]  p=0.2
    [ 2]  p=0.2
    [ 3]  p=0.2
    [--]  p=0.4
  [ 4]  p=0.15   (children mutually exclusive)
    [ 5]  p=0.2
    [ 6]  p=0.2
    [ 7]  p=0.2
    [--]  p=0.4
  [ 8]  p=0.15   (children mutually exclusive)
    [ 9]  p=0.2
    [10]  p=0.2
    [11]  p=0.2
    [--]  p=0.4
  [12]  p=0.05
  [13]  p=0.05
  [14]  p=0.05
  [15]  p=0.05
  [16]  p=0.05
  [17]  p=0.05
  [18]  p=0.05
  [19]  p=0.05

scoreable edges (9):
[(0, 1), (0, 2), (0, 3), (4, 5), (4, 6), (4, 7), (8, 9), (8, 10), (8, 11)]


### What an edge actually is

An edge `(p, c)` is a claim about co-firing, and it is **asymmetric**. Containment makes
`P(parent fires | child fires) = 1` exactly, while the reverse is weak — a parent
splits its firing across several children, so `P(child | parent)` is small. This is why
the metrics score *reverse* coverage (child → parent) and not the forward direction.

The cell below measures both directions on real samples, against a true edge and a
non-edge, so the asymmetry is visible rather than asserted.


In [5]:
N_DEMO = 20_000     # the sampler is pure Python, so keep this modest
demo = tree.sample(N_DEMO)

parent, child, other = TRUE_EDGES[0][0], TRUE_EDGES[0][1], TRUE_EDGES[3][0]
fires = demo.bool()

print(f"edge under test: {parent} -> {child}   (non-edge for contrast: {other} -> {child})\n")
print(f"base rates          : P({parent}) = {fires[:, parent].float().mean():.4f}"
      f"   P({child}) = {fires[:, child].float().mean():.4f}")
print(f"forward  P({child}|{parent})   = {fires[fires[:, parent], child].float().mean():.4f}"
      "   <- weak: the parent splits across children")
print(f"reverse  P({parent}|{child})   = {fires[fires[:, child], parent].float().mean():.4f}"
      "   <- the edge signal")
print(f"reverse  P({other}|{child})   = {fires[fires[:, child], other].float().mean():.4f}"
      "   <- non-edge, just the base rate")

violations = int((fires[:, child] & ~fires[:, parent]).sum())
siblings = int((fires[:, TRUE_EDGES[0][1]] & fires[:, TRUE_EDGES[1][1]]).sum())
print(f"\nchild firing with parent silent : {violations}   (containment)")
print(f"exclusive siblings co-firing    : {siblings}   (mutual exclusion)")


edge under test: 0 -> 1   (non-edge for contrast: 4 -> 1)

base rates          : P(0) = 0.1470   P(1) = 0.0298
forward  P(1|0)   = 0.2024   <- weak: the parent splits across children
reverse  P(0|1)   = 1.0000   <- the edge signal
reverse  P(4|1)   = 0.1529   <- non-edge, just the base rate

child firing with parent silent : 0   (containment)
exclusive siblings co-firing    : 0   (mutual exclusion)


### A sampler that matches training

The calibration below needs a lot of samples, and `Tree.sample` is pure Python — one
recursive walk per row. The vectorised version below draws all rows at once.

It is not just a speed rewrite. `metrics/validation/calibrate_on_trained_toy.py` carries
its own vectorised sampler which handles exclusive children differently: it picks a child
with `multinomial` weighted by that child's probability, then applies the same probability
*again* as a Bernoulli draw. Upstream's `Tree.sample` forces the picked child active
(`force_active=True`), so its probability is used once. The result is a child firing rate
about 5x lower there than in the data these SAEs were trained on.

Since the whole point is to grade an SAE on its own distribution, the sampler here follows
upstream, and the next cell checks it against `Tree.sample` rather than assuming it.


In [6]:
@torch.no_grad()
def sample_tree(root, n, gen=None):
    """[n, n_features] binary firings, vectorised, matching `Tree.sample` semantics.

    Containment: a node is only offered a draw on rows where its parent fired.
    Exclusion:   among mutually exclusive children exactly one is picked per row and
                 fires with certainty -- its `active_prob` is the pick weight, and is
                 deliberately NOT applied a second time.
    """
    out = torch.zeros(n, root.n_features)

    def rec(node, active):
        if node.index is not False:
            out[active, node.index] = 1.0
        if not node.children:
            return
        if node.mutually_exclusive_children:
            probs = torch.tensor([c.active_prob for c in node.children])
            pick = torch.multinomial(probs, n, replacement=True, generator=gen)
            for i, child in enumerate(node.children):
                rec(child, active & (pick == i))
        else:
            for child in node.children:
                rec(child, active & (torch.rand(n, generator=gen) < child.active_prob))

    rec(root, torch.rand(n, generator=gen) < root.active_prob)
    return out


# Check it against the reference implementation instead of trusting it.
n_check = 20_000
ref = tree.sample(n_check).mean(0)
fast = sample_tree(tree, n_check).mean(0)
worst = (ref - fast).abs().max().item()

print("feature | Tree.sample | sample_tree")
for i in range(tree.n_features):
    print(f"  {i:>5} |    {ref[i]:.4f}   |   {fast[i]:.4f}")
# Compare against the sampling error itself, not an eyeballed tolerance: the
# 1-sigma spread of a difference between two independent rate estimates.
se = (2 * ref * (1 - ref) / n_check).sqrt().max().item()
print(f"\nlargest fire-rate gap: {worst:.4f}   largest 1-sigma sampling error: {se:.4f}")
# Tolerance scaled to the noise, not to a round number: this is the largest of 20
# comparisons, so ~3 sigma turns up routinely and a fixed 0.01 would flake.
assert worst < 5 * se, "vectorised sampler disagrees with Tree.sample beyond sampling error"


feature | Tree.sample | sample_tree
      0 |    0.1440   |   0.1496
      1 |    0.0286   |   0.0308
      2 |    0.0286   |   0.0288
      3 |    0.0291   |   0.0302
      4 |    0.1539   |   0.1539
      5 |    0.0313   |   0.0314
      6 |    0.0302   |   0.0298
      7 |    0.0305   |   0.0325
      8 |    0.1470   |   0.1476
      9 |    0.0288   |   0.0291
     10 |    0.0293   |   0.0297
     11 |    0.0295   |   0.0317
     12 |    0.0474   |   0.0492
     13 |    0.0514   |   0.0470
     14 |    0.0511   |   0.0510
     15 |    0.0518   |   0.0483
     16 |    0.0490   |   0.0510
     17 |    0.0510   |   0.0485
     18 |    0.0485   |   0.0480
     19 |    0.0496   |   0.0485

largest fire-rate gap: 0.0056   largest 1-sigma sampling error: 0.0036


## 2. Matching latents to features

An SAE has no reason to learn feature 3 in slot 3. The dictionary is unordered, so
before any learned latent can be compared to a ground-truth feature, the two have to be
paired up.

`get_latent_perm` does that with a **Hungarian assignment** (`linear_sum_assignment`):
it builds a similarity matrix between latent activations and true activations over a
sample, converts it to a cost, and finds the one-to-one pairing with the lowest total
cost. Greedy nearest-neighbour matching would let two features claim the same latent;
the assignment forbids that.

The permutation it returns is used purely for *display and scoring* — it never touches
training. Everything downstream that indexes `[perm]` is putting the learned dictionary
into ground-truth order so a heatmap's diagonal is meaningful.


In [7]:
from scipy.optimize import linear_sum_assignment

@torch.no_grad()
def get_latent_perm(sae, tree_ds, include_all_latents=False):
    '''
    Get permutation of sae latents that tries to assign each latent to its closest-matching ground-truth feature
    H.T. Julian D'Costa for the linear_sum_assignment idea here.
    '''
    global DEVICE
    sample = tree_ds.tree.sample(1000).to(DEVICE)
    x = sample @ tree_ds.true_feats.to(DEVICE)

    feat_norms = tree_ds.true_feats.norm(dim=-1).to(DEVICE)
    scaled_sample = sample * feat_norms[None, :]

    true_acts = scaled_sample
    true_acts = true_acts/true_acts.max(dim=0, keepdim=True).values.clamp(min=1e-10)

    with torch.no_grad():
        sae_acts = sae.get_acts(x)
        sae_acts = sae_acts/sae_acts.max(dim=0, keepdim=True).values.clamp(min=1e-10)
        sims = (sae_acts.T @ true_acts).cpu()

    max_value = sims.max()
    cost_matrix = max_value - sims

    row_ind, col_ind = row_ind, col_ind = linear_sum_assignment(cost_matrix.detach().cpu().numpy().T)
    leftover_features = np.array(list({i for i in range(sims.shape[0]) if sae_acts.max(dim=0).values[i] > 0} - set(col_ind))).astype(int)

    if include_all_latents:
        return torch.tensor(np.concatenate([col_ind, leftover_features]))
    else:
        return torch.tensor(col_ind)

### The two dictionaries

Both SAEs come from the same class and differ in exactly two entries: `n_prefixes`
(10 nested prefixes vs. a single one spanning the whole dictionary) and
`permute_latents`. That is the entire experimental contrast.

`n_latents = tree.n_features` makes the dictionary exactly as large as the truth — no
spare capacity to hide a missed feature in — and `target_l0` is the true average number
of features firing at once, so the sparsity controller is aiming at the right answer.


In [8]:
matryoshka_config = {
    'n_latents': tree.n_features,
    'target_l0': 1.2338, # L0 of the true features
    'n_prefixes': 10,
    'd_model': D_MODEL,
    'n_steps': 40_000, # You could try fewer steps if this runs too slow for your taste; 15K e.g.
    'lr': 3e-2,
    'permute_latents': True,
    'sparsity_type': 'l1',
    'starting_sparsity_loss_scale': 0.2
}


vanilla_config = matryoshka_config | {'n_prefixes': 1, 'permute_latents': False}

## 3. Feature directions and the data stream

The firing vector still has to become an activation vector. Each feature is assigned a
direction in `d_model` dimensions and a sample is the sum of the directions that fired.

Here `d_model == n_features`, so the directions are a full orthogonal basis and there is
**no superposition** — the toy isolates the hierarchy question from the overcompleteness
question. Upstream draws a random orthogonal basis via QR; this notebook uses the
identity basis instead, which is only a rotation away but makes each feature a known
axis, so a saved checkpoint can be graded without also shipping the embedding it was
trained on. Feature norms still get a ±5% jitter so nothing depends on them being equal.

`TreeDataset` then streams batches forever: each `__getitem__` draws a fresh sample from
the tree, so the SAE never sees the same batch twice and there is no train/test split to
worry about.


In [9]:

from torch.utils.data import DataLoader
from sae import MatryoshkaSAE
from tqdm import tqdm

SEED = 0
torch.manual_seed(SEED)

# Ground-truth features are the identity basis, the convention the metrics repo
# calibrates against (validation/calibrate_on_trained_toy.py uses torch.eye).
# Upstream draws random orthogonal directions via QR instead, but with
# d_model == n_features that is only a rotation of this, so nothing is lost --
# and an axis-aligned ground truth means a checkpoint can be graded without
# shipping the embedding it was trained on.
true_feats = torch.eye(tree.n_features, D_MODEL, device=DEVICE)


# randomly scale the norms of features
random_scaling = 1+torch.randn(tree.n_features, device=DEVICE)* 0.05
true_feats *= random_scaling[:, None]


# Setup dataloader
dataset = TreeDataset(tree, true_feats.cpu(), batch_size=200, num_batches=vanilla_config['n_steps'])
dataloader = DataLoader(dataset, batch_size=None, num_workers=6, pin_memory=True)

# Create a reference batch to visualize vanilla and matryoshka with over the course of training
ref_acts = dataset.tree.sample(100).to(DEVICE)
ref_x = ref_acts @ dataset.true_feats.to(DEVICE)
ref_acts = ref_acts*random_scaling[None]


## 4. Training both SAEs

Both models step on the same batch, so any difference between them is the nesting and
not the data. The live heatmaps redraw every 150 steps — rows are samples, columns are
latents in ground-truth order.

What to watch: the top panel is the truth. A healthy Matryoshka panel converges toward
it. The vanilla panel typically shows the failure this toy was built to expose — a
column that fires whenever *any* member of a family fires (a parent absorbed into a
broad latent), or a parent column that has gone dark because its children swallowed it.

`sparsity_controller` is an adaptive L1 weight chasing `target_l0 = 1.2338`, the true
average number of active features. It is printed in each title; expect it to move early
and settle.


In [10]:
from IPython.display import clear_output
from heatmap import heatmap


vanilla_sae = MatryoshkaSAE(**vanilla_config).to(DEVICE)
matryoshka_sae = MatryoshkaSAE(**matryoshka_config).to(DEVICE)

for step, batch in tqdm(enumerate(dataloader), total=vanilla_config['n_steps']):
    batch = batch.to(DEVICE)

    matryoshka_sae.step(batch)
    vanilla_sae.step(batch)

    if step % 150 == 0:
        clear_output(wait=True)

        heatmap(ref_acts.cpu(), title='Ground-Truth Features').show()

        matryoshka_perm = get_latent_perm(matryoshka_sae, dataset)
        heatmap(matryoshka_sae.get_acts(ref_x)[:,matryoshka_perm].cpu(), title=f'Matryoshka Latents  |  Sparsity Reg: {matryoshka_sae.sparsity_controller():.2f}  |  Step {step}',).show()

        vanilla_perm = get_latent_perm(vanilla_sae, dataset)
        heatmap(vanilla_sae.get_acts(ref_x)[:,vanilla_perm].cpu(), title=f'Vanilla Latents  |  Sparsity Reg: {vanilla_sae.sparsity_controller():.2f}  |  Step {step}').show()

100%|██████████| 40000/40000 [02:58<00:00, 223.74it/s]


## 5. Saving

The upstream `MatryoshkaSAE` has no `save`/`load`, so without this cell the weights only
ever live in kernel memory. `true_feats` and the tree are saved alongside them: without
the directions a checkpoint was trained on, nothing downstream can tell which latent
recovered which feature.

Checkpoints go to **`metrics/outputs/toy_trained/`** — the directory
`validation/calibrate_on_trained_toy.py` reads — rather than to `sae-training/checkpoints/`,
where they landed while this notebook lived in that repo. One directory per run, each with
`cfg.json` + `sae_weights.safetensors`; the `.pt` files sit beside them and are gitignored
(`outputs/**/*.pt`), while the safetensors directories are tracked.

> **This overwrites the graded reference.** `calibrate_on_trained_toy.py` defaults to
> `outputs/toy_trained/matryoshka_toy/`, and the committed weights there are from a
> *different* run than a fresh execution of this notebook produces — same config, different
> numbers (the committed one recovers 6 of 9 edges; this notebook's last run recovered 9).
> Re-running this cell replaces the checkpoint the paper's Tier-2 numbers are read from.
> `git restore outputs/toy_trained/matryoshka_toy/` puts the old one back, and
> `SAVE_AS` below renames this run's output if you would rather keep both.

In [11]:
import torch

# Where a run lands. The name is the directory `calibrate_on_trained_toy.py` grades by
# default, so writing it replaces the graded reference -- change SAVE_AS (e.g. to
# "matryoshka_toy_rerun") to keep this run beside the old one instead of over it, then
# grade it with EXP0_TOY_CKPT=outputs/toy_trained/<name>.
SAVE_AS = {"matryoshka": "matryoshka_toy", "vanilla": "vanilla_toy"}

# METRICS_ROOT comes from the setup cell, so this stays correct wherever the notebook
# sits. It used to be SAE_TRAINING/"checkpoints" -- correct while this file lived in
# that repo, and a checkpoint nothing in the metrics pipeline reads ever since.
ckpt_dir = METRICS_ROOT / "outputs" / "toy_trained"
ckpt_dir.mkdir(parents=True, exist_ok=True)

for _name in SAVE_AS.values():
    if (ckpt_dir / _name / "sae_weights.safetensors").exists():
        print(f"note: overwriting {ckpt_dir / _name} — `git restore` it to undo")

# true_feats/random_scaling make the checkpoint self-describing: without the
# directions it was trained on, nothing downstream can tell which latent recovered
# which feature. They are the identity basis here, but saved rather than assumed.
extras = {'true_feats': true_feats.cpu(), 'random_scaling': random_scaling.cpu(),
          'tree_dict': tree_dict, 'seed': SEED}

torch.save({'config': matryoshka_config, 'state_dict': matryoshka_sae.state_dict(), **extras},
           ckpt_dir / f"{SAVE_AS['matryoshka']}.pt")
torch.save({'config': vanilla_config, 'state_dict': vanilla_sae.state_dict(), **extras},
           ckpt_dir / f"{SAVE_AS['vanilla']}.pt")

print('saved to', ckpt_dir)


# --- also write the layout the metrics repo reads -------------------------------
# `cfg.json` + `sae_weights.safetensors`, as produced by
# architectures/base.py::save_pretrained and consumed by
# validation/calibrate_on_trained_toy.py::load_sae. safetensors stores tensors only,
# so the config, the tree and the seed go into the JSON sidecar beside it.
import json as _json
from safetensors.torch import save_file


def save_safetensors(sae, config, name):
    out_dir = ckpt_dir / name
    out_dir.mkdir(parents=True, exist_ok=True)

    tensors = {k: v.detach().contiguous().cpu() for k, v in sae.state_dict().items()}
    tensors['true_feats'] = true_feats.detach().contiguous().cpu()
    tensors['random_scaling'] = random_scaling.detach().contiguous().cpu()
    save_file(tensors, str(out_dir / 'sae_weights.safetensors'))

    (out_dir / 'cfg.json').write_text(_json.dumps({
        'd_in': config['d_model'],
        'd_sae': config['n_latents'],
        'activation_function': 'relu',      # upstream MatryoshkaSAE is ReLU + adaptive L1
        'k': None,                          # unused for relu
        'architecture': 'MatryoshkaSAE',
        'notebook_config': config,
        'tree_dict': tree_dict,
        'seed': SEED,
    }, indent=2))
    print('saved', out_dir)


save_safetensors(matryoshka_sae, matryoshka_config, SAVE_AS['matryoshka'])
save_safetensors(vanilla_sae, vanilla_config, SAVE_AS['vanilla'])

note: overwriting /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/metrics/outputs/toy_trained/matryoshka_toy — `git restore` it to undo
saved to /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/metrics/outputs/toy_trained
saved /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/metrics/outputs/toy_trained/matryoshka_toy
saved /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/metrics/outputs/toy_trained/vanilla_toy


## 6. Reading the cosine heatmaps

Four matrices, all in ground-truth order via the permutation from §2, so **a perfect
recovery is a bright diagonal**.

- *Ground-truth cosine similarity* — the identity, by construction. It is the reference
  for what "recovered" looks like.
- *Decoder vs. ground truth* — what each latent writes back into the residual.
- *Encoder vs. ground truth* — what each latent reads. Encoder and decoder disagreeing
  is itself informative: a latent that reads a family but writes one member is absorbing.

Off-diagonal structure is the interesting part. A bright block where a parent meets its
children is the collapse this notebook is testing for.


In [12]:
import torch.nn.functional as F

matryoshka_perm = get_latent_perm(matryoshka_sae, dataset)
vanilla_perm = get_latent_perm(vanilla_sae, dataset)

# Compute cosine similarity matrix of ground truth features
gt_cosine = F.normalize(true_feats, dim=1) @ F.normalize(true_feats, dim=1).T
gt_sims = heatmap(gt_cosine.cpu(), title='Ground Truth Feature Cosine Similarity')
gt_sims.show()

# Compute cosine similarity between learned and ground truth features
matryoshka_feats = matryoshka_sae.W_dec.data
vanilla_feats = vanilla_sae.W_dec.data

matryoshka_cosine = F.normalize(matryoshka_feats, dim=1)[matryoshka_perm] @ F.normalize(true_feats, dim=1).T
vanilla_cosine = F.normalize(vanilla_feats, dim=1)[vanilla_perm] @ F.normalize(true_feats, dim=1).T

m_dec_sims = heatmap(matryoshka_cosine.cpu(), title='Matryoshka Decoder, Ground-Truth Cosine Similarity', dim_names=('Matryoshka', 'True Feature'))
m_dec_sims.show()

v_dec_sims = heatmap(vanilla_cosine.cpu(), title='Vanilla Decoder, Ground-Truth Cosine Similarity', dim_names=('Vanilla', 'Ground-Truth'))
v_dec_sims.show()

# Compare encoder and decoder weights
matryoshka_enc = matryoshka_sae.W_enc.data.T
vanilla_enc = vanilla_sae.W_enc.data.T

matryoshka_enc_true = F.normalize(matryoshka_enc, dim=1)[matryoshka_perm] @ F.normalize(true_feats, dim=1).T
vanilla_enc_true = F.normalize(vanilla_enc, dim=1)[vanilla_perm] @ F.normalize(true_feats, dim=1).T


m_enc_sims = heatmap(matryoshka_enc_true.cpu(), title='Matryoshka Encoder, Ground-Truth Cosine Similarity', dim_names=('Latent', 'Feature'))
m_enc_sims.show()
v_enc_sims = heatmap(vanilla_enc_true.cpu(), title='Vanilla Encoder, Ground-Truth Cosine Similarity', dim_names=('Latent', 'Feature'))
v_enc_sims.show()


## 7. Feature recovery, calibrated against a null

The heatmaps above are read by eye. This reduces them to one number per SAE: of the
ground-truth features, how many did the SAE actually find? `get_latent_perm` already
pairs each true feature with its best-matching latent, so recovery is the cosine
similarity along that pairing.

A fixed cutoff like 0.9 is arbitrary and depends on `d_model` — in 20 dimensions two
random directions already sit around 1/sqrt(20) ~ 0.22 apart. So the cutoff is calibrated
against a null instead, following the same convention as `metrics/independence_null.py`:
draw random decoder directions, run them through the *same* Hungarian assignment, and take
a high quantile of the resulting matched cosines. The null matches on decoder similarity
directly, which is the very quantity being thresholded, so it is conservative by design.


In [13]:
NULL_TRIALS = 200
NULL_QUANTILE = 0.99

gt_dirs = F.normalize(true_feats, dim=1)


@torch.no_grad()
def null_matched_cosines(n_latents, n_trials=NULL_TRIALS):
    """Matched-cosine distribution when the decoder carries no information.

    Includes the best-of-n_latents selection effect that a plain random-pair
    baseline would miss.
    """
    out = []
    for _ in range(n_trials):
        rand_dec = F.normalize(torch.randn_like(torch.empty(n_latents, gt_dirs.shape[1],
                                                            device=gt_dirs.device)), dim=1)
        sims = rand_dec @ gt_dirs.T                                # [n_latents, n_features]
        _, col = linear_sum_assignment((sims.max() - sims).cpu().numpy().T)
        out.append((rand_dec[col] * gt_dirs).sum(-1))
    return torch.cat(out)


null_cos = null_matched_cosines(matryoshka_sae.n_latents)
threshold = null_cos.quantile(NULL_QUANTILE).item()

print(f"null: {NULL_TRIALS} trials, mean cos {null_cos.mean():.3f}, "
      f"max {null_cos.max():.3f}")
print(f"calibrated threshold (q{NULL_QUANTILE:.2f} of null) = {threshold:.3f}\n")


def feature_recovery(sae, perm, label):
    matched_cos = (F.normalize(sae.W_dec.data, dim=1)[perm] * gt_dirs).sum(-1)

    n_recovered = (matched_cos >= threshold).sum().item()
    n_strict = (matched_cos >= 0.9).sum().item()
    n_alive = (sae.get_acts(ref_x).max(dim=0).values > 0).sum().item()
    n_gt = len(gt_dirs)

    print(f"{label:>10} | recovered {n_recovered}/{n_gt} = {n_recovered / n_gt:>6.1%}"
          f" | at cos>=0.9: {n_strict}/{n_gt}"
          f" | mean cos {matched_cos.mean():.3f}"
          f" | alive latents {n_alive}/{sae.n_latents}")
    return matched_cos


m_cos = feature_recovery(matryoshka_sae, matryoshka_perm, "Matryoshka")
v_cos = feature_recovery(vanilla_sae, vanilla_perm, "Vanilla")

print(f"\nmissed by Matryoshka: {(m_cos < threshold).nonzero().flatten().tolist()}")
print(f"missed by Vanilla:    {(v_cos < threshold).nonzero().flatten().tolist()}")


null: 200 trials, mean cos 0.376, max 0.790
calibrated threshold (q0.99 of null) = 0.665

Matryoshka | recovered 20/20 = 100.0% | at cos>=0.9: 20/20 | mean cos 0.963 | alive latents 20/20
   Vanilla | recovered 19/20 =  95.0% | at cos>=0.9: 11/20 | mean cos 0.868 | alive latents 20/20

missed by Matryoshka: []
missed by Vanilla:    [5]


## 8. Edge recovery: do the hierarchy metrics rebuild the tree?

Everything above scores *features*. This scores *edges* — the actual Exp-0 question.
It reuses the production metrics from the `metrics` repo unchanged and grades the edges
they keep against the known parent->child tree, exactly as
`metrics/validation/calibrate_on_trained_toy.py` does.

The ground-truth basis now matches that script's `torch.eye(F)` assumption, so the
two are directly comparable. One deliberate difference remains, because it grades a
*different* checkpoint (`outputs/toy_trained/`, batch_topk k=2, trained on GPU 3):

- its `encode()` reimplements a batch_topk forward pass. The SAE here is ReLU with a
  running-average input normalizer, so the model's own `get_acts` is called rather than
  a reimplementation that would silently drop the normalizer.


In [14]:
import sys

# METRICS_ROOT comes from the setup cell -- this notebook lives inside that repo now,
# so it is found rather than guessed from the workspace layout. Only the metric
# functions are imported: the tree and the sampler are this notebook's own, so the
# SAE is graded on exactly the distribution it was trained on.
if str(METRICS_ROOT) not in sys.path:
    sys.path.insert(0, str(METRICS_ROOT))

from metrics import (
    coverage_legs, keep_edges, edge_reconstruction_condition,
    frequency_controlled_coverage, frequency_buckets,
)
from metrics.reconstruction import per_token_ablation_gain

CALIB_N = 200_000
MIN_MATCH_COS = 0.4          # same latent->feature cutoff the calibration script uses
TRUTH = set(TRUE_EDGES)


@torch.no_grad()
def sae_acts_and_resid(sae, x):
    """Activations from the model's own path, plus the reconstruction residual.

    get_acts applies the running-average normalizer and scales by ||W_dec||, so the
    reconstruction has to be rebuilt in that same normalized space.
    """
    x_norm = sae.normalizer.normalize(x, update=False)
    codes = F.relu(x_norm @ sae.W_enc + sae.b_enc)
    x_hat = sae.normalizer.unnormalize(codes @ sae.W_dec + sae.b_dec)
    return sae.get_acts(x), x - x_hat


def match_latents(sae, true_dirs, min_cos=MIN_MATCH_COS):
    """Each latent -> the true feature its decoder points at (-1 if none)."""
    cos = F.normalize(sae.W_dec.data, dim=1) @ F.normalize(true_dirs, dim=1).T
    best = cos.argmax(dim=1)
    best[cos.max(dim=1).values < min_cos] = -1
    return best


def calibrate(sae, label):
    gen = torch.Generator().manual_seed(SEED)
    gt = sample_tree(tree, CALIB_N, gen)                  # same process the SAE saw
    x = gt.to(DEVICE) @ true_feats
    acts, resid = sae_acts_and_resid(sae, x)

    m = match_latents(sae, true_feats).tolist()
    recovered = {t for t in m if t >= 0}

    fired = (acts > 1e-3).double().cpu()
    g = per_token_ablation_gain(acts.double().cpu(), resid.double().cpu(),
                                sae.W_dec.data.double().cpu())
    err = (resid.double().cpu() ** 2).sum(dim=1)

    parents = {p for p, _ in TRUTH}
    children = {c for _, c in TRUTH}
    parent_lat = [i for i, t in enumerate(m) if t in parents]
    child_lat = [i for i, t in enumerate(m) if t in children]

    print(f"--- {label} ---")
    print(f"recovered {len(recovered)}/{tree.n_features} features; latents matched to "
          f"parents: {len(parent_lat)}, to children: {len(child_lat)}")
    if not parent_lat or not child_lat:
        print("did not recover both parents and children; cannot score edges.\n")
        return None

    fp_, fc_ = fired[:, parent_lat], fired[:, child_lat]
    fire_p, fire_c = fp_.sum(0), fc_.sum(0)

    # Gate 1: reverse coverage -- does the child fire almost only when the parent does?
    R, _ = coverage_legs(fp_.T @ fc_, fire_p, fire_c)
    edge_mask = keep_edges(R, fire_p, fire_c, 0.5, 20)

    # Gate 2: does ablating the parent actually hurt the child's reconstruction?
    recon = edge_reconstruction_condition(
        fc_.T @ err, g[:, parent_lat].T @ fc_, (fc_ * g[:, child_lat]).sum(0), 0.01)

    # Gate 3: does the edge survive once token frequency is controlled for?
    token_ids = fired.argmax(dim=1).long()
    counts = torch.zeros(int(token_ids.max()) + 1, dtype=torch.float64)
    counts.scatter_add_(0, token_ids, torch.ones(CALIB_N, dtype=torch.float64))
    buckets = frequency_buckets(counts, 0.5, 0.4)[token_ids]
    cbb = torch.zeros(3, len(parent_lat), len(child_lat), dtype=torch.float64)
    fcb = torch.zeros(3, len(child_lat), dtype=torch.float64)
    for k in range(3):
        sel = (buckets == k).double().unsqueeze(1)
        cbb[k] = fp_.T @ (fc_ * sel)
        fcb[k] = (fc_ * sel).sum(0)
    fcov = frequency_controlled_coverage(cbb, fcb, edge_mask)

    survivors = edge_mask & recon["passes"] & (fcov["survival"] >= 0.5)
    found = {(m[parent_lat[pi]], m[child_lat[ci]])
             for pi in range(survivors.shape[0]) for ci in range(survivors.shape[1])
             if survivors[pi, ci]}

    tp, fp_edges, fn = found & TRUTH, found - TRUTH, TRUTH - found
    precision = len(tp) / max(len(found), 1)
    recall = len(tp) / max(len(TRUTH), 1)

    print(f"true positives {len(tp)}/{len(TRUTH)}  false pos {len(fp_edges)}  "
          f"false neg {len(fn)}")
    if fn:
        print(f"  missed   : {sorted(fn)}")
    if fp_edges:
        print(f"  spurious : {sorted(fp_edges)}")
    print(f"precision {precision:.2f}   recall {recall:.2f}   "
          f"VERDICT: {'PASS' if precision >= 0.8 and recall >= 0.8 else 'NEEDS WORK'}\n")

    return {"precision": precision, "recall": recall, "found": sorted(found),
            "recovered_features": sorted(recovered)}


calib_matryoshka = calibrate(matryoshka_sae, "Matryoshka")
calib_vanilla = calibrate(vanilla_sae, "Vanilla")


--- Matryoshka ---
recovered 20/20 features; latents matched to parents: 3, to children: 9
true positives 9/9  false pos 0  false neg 0
precision 1.00   recall 1.00   VERDICT: PASS

--- Vanilla ---
recovered 17/20 features; latents matched to parents: 6, to children: 6
true positives 0/9  false pos 0  false neg 9
  missed   : [(0, 1), (0, 2), (0, 3), (4, 5), (4, 6), (4, 7), (8, 9), (8, 10), (8, 11)]
precision 0.00   recall 0.00   VERDICT: NEEDS WORK



### What "recovering an edge" looks like

The numbers above are a summary; this is the same tree with each edge coloured by what
the metrics did with it. Read it against the ground-truth drawing in §1 — same layout,
same node positions.

- **green** — a true edge that survived all three gates (true positive);
- **red, dashed** — a true edge the metrics dropped (false negative);
- **purple, dotted** — a pair the metrics kept that is *not* in the tree (false positive);
  these are drawn as new links, so they cut across the layout rather than following it;
- **hollow grey node** — a feature no latent recovered, so no edge touching it could be
  found even in principle.

Precision is the share of coloured-in links that are green; recall is the share of the
tree's nine edges that came out green.


In [15]:
def draw_recovery(result, label):
    """The tree of §1, with every edge coloured by the calibration's verdict."""
    if result is None:
        print(f"{label}: nothing to draw — the SAE did not recover both endpoints.")
        return None

    nodes, links, n_cols = tree_layout(tree)
    xy_of = {n.index: xy for xy, n in nodes if n.index is not False}
    found = {tuple(e) for e in result["found"]}
    recovered = set(result["recovered_features"])

    fig = go.Figure()

    # Structural links first, so they sit behind the verdicts.
    for (x0, y0), (x1, y1), parent, child in links:
        if (parent.index, child.index) in TRUTH:
            continue
        fig.add_trace(go.Scatter(x=[x0, x1], y=[y0, y1], mode="lines", hoverinfo="skip",
                                 showlegend=False, line=dict(color=GREY, width=1, dash="dot")))

    for (p, c) in TRUE_EDGES:                      # true positives and false negatives
        hit = (p, c) in found
        (x0, y0), (x1, y1) = xy_of[p], xy_of[c]
        fig.add_trace(go.Scatter(
            x=[x0, x1], y=[y0, y1], mode="lines", showlegend=False,
            line=dict(color=GREEN if hit else RED, width=3 if hit else 2,
                      dash="solid" if hit else "dash"),
            hovertext=f"{p} -> {c}: {'recovered' if hit else 'missed'}", hoverinfo="text"))

    for (p, c) in sorted(found - TRUTH):            # false positives
        # Bowed, not straight: a spurious pair often joins two nodes on the same row,
        # and a straight line there would run through the nodes in between and read
        # as a chain of edges that does not exist.
        xs, ys = _arc(xy_of[p], xy_of[c])
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode="lines", showlegend=False,
            line=dict(color=PURPLE, width=2, dash="dot", shape="spline"),
            hovertext=f"{p} -> {c}: spurious", hoverinfo="text"))

    for is_rec in (True, False):
        pts = [(xy, n) for xy, n in nodes
               if n.index is not False and (n.index in recovered) == is_rec]
        if not pts:
            continue
        fig.add_trace(go.Scatter(
            x=[p[0][0] for p in pts], y=[p[0][1] for p in pts],
            mode="markers+text", showlegend=False,
            text=[str(n.index) for _, n in pts], textposition="middle center",
            textfont=dict(color="white" if is_rec else GREY, size=11),
            marker=dict(size=26, color=BLUE if is_rec else "white",
                        line=dict(width=1.5, color=BLUE if is_rec else GREY)),
            hovertext=[("feature recovered by a latent" if is_rec
                        else "no latent recovered this feature") for _ in pts],
            hoverinfo="text"))

    _legend(fig,
            lines=[("recovered (true positive)", dict(color=GREEN, width=3)),
                   ("missed (false negative)", dict(color=RED, width=2, dash="dash")),
                   ("spurious (false positive)", dict(color=PURPLE, width=2, dash="dot"))],
            markers=[("feature recovered", dict(size=13, color=BLUE, line=dict(width=1.5, color=BLUE))),
                     ("feature not recovered", dict(size=13, color="white", line=dict(width=1.5, color=GREY)))])

    n_tp = len(found & TRUTH)
    return _finish(fig, f"{label} — {n_tp}/{len(TRUTH)} edges recovered   "
                        f"(precision {result['precision']:.2f}, "
                        f"recall {result['recall']:.2f})", nodes, n_cols)


for _res, _label in [(calib_matryoshka, "Matryoshka"), (calib_vanilla, "Vanilla")]:
    _fig = draw_recovery(_res, _label)
    if _fig is not None:
        _fig.show()


### One figure: the tree, before, and after

The listing, the ground truth and the outcome side by side. The panels reuse the very
same traces as the figures above rather than redrawing them, so the three can never
disagree about what the tree is.


In [16]:
from plotly.subplots import make_subplots


def draw_summary(result, label):
    """Text listing | ground-truth tree | tree after the battery, as one figure."""
    if result is None:
        print(f"{label}: nothing to draw — the SAE did not recover both endpoints.")
        return None

    truth_fig = draw_tree(tree, set(TRUE_EDGES))
    recovery_fig = draw_recovery(result, label)
    _, _, n_rows = tree_layout(tree)

    fig = make_subplots(rows=1, cols=3, column_widths=[0.26, 0.37, 0.37],
                        horizontal_spacing=0.02,
                        subplot_titles=("the tree, exactly",
                                        "ground truth",
                                        f"after the battery — {label}"))

    # Copy the traces instead of rebuilding them. Legend-only proxies carry x=[None];
    # they are added once, outside the panels, so the key is not printed three times.
    for src, col in ((truth_fig, 2), (recovery_fig, 3)):
        for trace in src.data:
            is_proxy = trace.x is None or trace.x[0] is None
            if is_proxy:
                if col == 3:                      # the outcome key is the informative one
                    fig.add_trace(trace, row=1, col=1)
            else:
                trace.showlegend = False
                fig.add_trace(trace, row=1, col=col)

    listing = tree_text(tree) + f"\n\nscoreable edges ({len(TRUE_EDGES)}):\n"
    listing += "\n".join(str(TRUE_EDGES[i:i + 3]) for i in range(0, len(TRUE_EDGES), 3))
    fig.add_annotation(text=listing.replace("\n", "<br>").replace(" ", "&nbsp;"),
                       xref="x domain", yref="y domain", x=0, y=1,
                       xanchor="left", yanchor="top", showarrow=False, align="left",
                       font=dict(family="monospace", size=10, color="#2B2B33"),
                       row=1, col=1)

    for axis in fig.layout:
        if axis.startswith(("xaxis", "yaxis")):
            fig.layout[axis].visible = False
    for col in (2, 3):
        fig.update_xaxes(range=[-0.35, 2.35], row=1, col=col)
        fig.update_yaxes(range=[-n_rows + 0.4, 0.6], row=1, col=col)
    fig.update_xaxes(range=[0, 1], row=1, col=1)
    fig.update_yaxes(range=[-n_rows + 0.4, 0.6], row=1, col=1)

    for note in fig.layout.annotations[:3]:
        note.font = dict(size=12, color="#2B2B33")
        note.xanchor = "left"
        note.x = note.x - 0.12

    fig.update_layout(width=1300, height=int(185 + 34 * n_rows),
                      plot_bgcolor="white", showlegend=True,
                      margin=dict(l=20, r=20, t=55, b=105),
                      legend=dict(orientation="h", yanchor="top", y=-0.02, x=0))
    return fig


for _res, _label in [(calib_matryoshka, "Matryoshka"), (calib_vanilla, "Vanilla")]:
    _fig = draw_summary(_res, _label)
    if _fig is not None:
        _fig.show()


## 9. Exporting the figures

Every figure this notebook draws, written to `metrics/validation/figures/tier2/`
as both `.html` (interactive, hover intact) and `.png` (needs `kaleido`).

They used to go to `figures/` relative to the working directory, which the setup
cell had pointed at the cloned `matryoshka-saes` repo: our results landed inside a
third-party checkout, overwrote five files that repo tracks, and would have
vanished on the next re-clone. `FIG_DIR` from the setup cell is an absolute path
inside this repo instead, beside the Tier-1 notebook's own `figures/`.

The three training heatmaps are rebuilt here from the trained SAEs rather than
captured inside the training loop, so nothing has to be retrained to export them.

In [17]:
# Every figure in the notebook, in the order it appears above. The drawing
# functions are called again rather than the loop-time figures being stashed:
# `heatmap`, `draw_tree`, `draw_recovery` and `draw_summary` are all still in
# memory, so this costs a redraw and not a retrain.
figures = {
    # 1. the world
    "toy_tree_declared": draw_tree(tree, set(TRUE_EDGES)),
    # 4. what the two SAEs learned, at the end of training rather than at the
    #    last multiple of 150 steps the live view happened to land on
    "latents_ground_truth": heatmap(ref_acts.cpu(), title="Ground-Truth Features"),
    "latents_matryoshka": heatmap(
        matryoshka_sae.get_acts(ref_x)[:, matryoshka_perm].cpu(),
        title=f"Matryoshka Latents  |  Sparsity Reg: "
              f"{matryoshka_sae.sparsity_controller():.2f}  |  final"),
    "latents_vanilla": heatmap(
        vanilla_sae.get_acts(ref_x)[:, vanilla_perm].cpu(),
        title=f"Vanilla Latents  |  Sparsity Reg: "
              f"{vanilla_sae.sparsity_controller():.2f}  |  final"),
    # 6. the cosine heatmaps
    "gt_feature_cosine": gt_sims,
    "matryoshka_decoder_gt_cosine": m_dec_sims,
    "vanilla_decoder_gt_cosine": v_dec_sims,
    "matryoshka_encoder_gt_cosine": m_enc_sims,
    "vanilla_encoder_gt_cosine": v_enc_sims,
}

# 8-9. edge recovery and the summary, one per SAE. Both return None when the SAE
# recovered no parents or no children, which is a real outcome for vanilla runs --
# skipped by name rather than crashing the export.
for _label, _res in (("matryoshka", calib_matryoshka), ("vanilla", calib_vanilla)):
    for _name, _fn in (("edge_recovery", draw_recovery), ("summary", draw_summary)):
        _f = _fn(_res, _label.capitalize())
        if _f is not None:
            figures[f"{_name}_{_label}"] = _f

for _name, _fig in figures.items():
    _fig.write_html(str(FIG_DIR / f"{_name}.html"))
    try:
        _fig.write_image(str(FIG_DIR / f"{_name}.png"), scale=4)
    except Exception as _exc:                      # kaleido missing or headless
        print(f"  PNG skipped for {_name}: {type(_exc).__name__}: {_exc}")

print(f"wrote {len(figures)} figures to {FIG_DIR}")
for _name in figures:
    print("  ", _name)

wrote 13 figures to /Users/ruqiya/Codeing-repos/Research/eleuther/soar-eleuther-i6-hierarchy/metrics/validation/figures/tier2
   toy_tree_declared
   latents_ground_truth
   latents_matryoshka
   latents_vanilla
   gt_feature_cosine
   matryoshka_decoder_gt_cosine
   vanilla_decoder_gt_cosine
   matryoshka_encoder_gt_cosine
   vanilla_encoder_gt_cosine
   edge_recovery_matryoshka
   summary_matryoshka
   edge_recovery_vanilla
   summary_vanilla
